In [1]:
import pandas as pd
import sqlite3

In [2]:
df = pd.read_csv(
    "processed/telco_eda_processed.csv"
)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TenureGroup,MonthlyChargeGroup,NumberOfServices,HighValueCustomer,CustomerSegment
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Yes,Electronic check,29.85,29.85,No,0-6 Months,Low,1,False,New Customer
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,Mailed check,56.95,1889.50,No,25-48 Months,Medium,3,False,Regular Customer
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Yes,Mailed check,53.85,108.15,Yes,0-6 Months,Medium,3,False,New Customer
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,Bank transfer (automatic),42.30,1840.75,No,25-48 Months,Medium,3,False,Regular Customer
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,Yes,Electronic check,70.70,151.65,Yes,0-6 Months,High,1,False,New Customer


In [3]:
connection = sqlite3.connect(
    "processed/customer_intelligence.db"
)

df.to_sql(
    "customers",
    connection,
    if_exists="replace",
    index=False
)

print("SQLite database created successfully.")

SQLite database created successfully.


In [4]:
query = """
SELECT COUNT(*) AS total_customers
FROM customers;
"""

pd.read_sql_query(query, connection)

,total_customers
0,7043


In [5]:
query = """
SELECT 
    Churn,
    COUNT(*) AS customer_count
FROM customers
GROUP BY Churn;
"""

pd.read_sql_query(query, connection)

,Churn,customer_count
0,No,5174
1,Yes,1869


In [6]:
query = """
SELECT
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate
FROM customers;
"""

pd.read_sql_query(query, connection)

,churn_rate
0,26.54


In [7]:
query = """
SELECT
    Contract,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate
FROM customers
GROUP BY Contract
ORDER BY churn_rate DESC;
"""

pd.read_sql_query(query, connection)

,Contract,total_customers,churned_customers,churn_rate
0,Month-to-month,3875,1655,42.71
1,One year,1473,166,11.27
2,Two year,1695,48,2.83


In [8]:
query = """
SELECT
    PaymentMethod,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate
FROM customers
GROUP BY PaymentMethod
ORDER BY churn_rate DESC;
"""

pd.read_sql_query(query, connection)

,PaymentMethod,total_customers,churned_customers,churn_rate
0,Electronic check,2365,1071,45.29
1,Mailed check,1612,308,19.11
2,Bank transfer (automatic),1544,258,16.71
3,Credit card (automatic),1522,232,15.24


In [9]:
query = """
SELECT
    InternetService,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate
FROM customers
GROUP BY InternetService
ORDER BY churn_rate DESC;
"""

pd.read_sql_query(query, connection)

,InternetService,total_customers,churned_customers,churn_rate
0,Fiber optic,3096,1297,41.89
1,DSL,2421,459,18.96
2,No,1526,113,7.40


In [10]:
query = """
SELECT
    ROUND(AVG(MonthlyCharges), 2) AS average_monthly_charges
FROM customers;
"""

pd.read_sql_query(query, connection)

,average_monthly_charges
0,64.76


In [11]:
query = """
SELECT
    ROUND(AVG(TotalCharges), 2) AS average_total_charges
FROM customers;
"""

pd.read_sql_query(query, connection)

,average_total_charges
0,2283.3


In [12]:
high_value_threshold = df["MonthlyCharges"].quantile(0.75)

print("High-value threshold:", round(high_value_threshold, 2))

High-value threshold: 89.85


In [13]:
query = f"""
SELECT
    customerID,
    tenure,
    Contract,
    MonthlyCharges,
    TotalCharges,
    Churn
FROM customers
WHERE MonthlyCharges >= {high_value_threshold}
ORDER BY MonthlyCharges DESC
LIMIT 20;
"""

pd.read_sql_query(query, connection)

,customerID,tenure,Contract,MonthlyCharges,TotalCharges,Churn
0,7569-NMZYQ,72,Two year,118.75,8672.45,No
1,8984-HPEMB,71,Two year,118.65,8477.60,No
2,5989-AXPUC,68,Two year,118.60,7990.05,No
3,5734-EJKXG,61,One year,118.60,7365.70,No
4,8199-ZLLSA,67,One year,118.35,7804.15,Yes
5,9924-JPRMC,72,Two year,118.20,8547.15,No
6,2889-FPWRM,72,One year,117.80,8684.80,Yes
7,3810-DVDQQ,72,Two year,117.60,8308.90,No
8,9739-JLPQJ,72,Two year,117.50,8670.10,No
9,2302-ANTDP,48,Month-to-month,117.45,5438.90,Yes


In [14]:
query = f"""
SELECT
    customerID,
    tenure,
    Contract,
    InternetService,
    MonthlyCharges,
    TotalCharges,
    Churn
FROM customers
WHERE MonthlyCharges >= {high_value_threshold}
AND Churn = 'Yes'
ORDER BY MonthlyCharges DESC;
"""

high_risk_customers = pd.read_sql_query(query, connection)

high_risk_customers.head(20)

,customerID,tenure,Contract,InternetService,MonthlyCharges,TotalCharges,Churn
0,8199-ZLLSA,67,One year,Fiber optic,118.35,7804.15,Yes
1,2889-FPWRM,72,One year,Fiber optic,117.80,8684.80,Yes
2,2302-ANTDP,48,Month-to-month,Fiber optic,117.45,5438.90,Yes
3,9053-JZFKV,67,Two year,Fiber optic,116.20,7752.30,Yes
4,1444-VVSGW,70,One year,Fiber optic,115.65,7968.85,Yes
5,0201-OAMXR,70,One year,Fiber optic,115.55,8127.60,Yes
6,4361-BKAXE,41,Month-to-month,Fiber optic,114.50,4527.45,Yes
7,1555-DJEQW,70,Two year,Fiber optic,114.20,7723.90,Yes
8,9158-VCTQB,41,Month-to-month,Fiber optic,113.60,4594.95,Yes
9,7279-BUYWN,41,Month-to-month,Fiber optic,113.20,4689.50,Yes


In [15]:
print("High-value churned customers:", len(high_risk_customers))

High-value churned customers: 580


In [16]:
query = """
SELECT
    customerID,
    tenure,
    Contract,
    MonthlyCharges,
    TotalCharges,
    Churn
FROM customers
WHERE tenure >= 48
ORDER BY tenure DESC;
"""

long_term_customers = pd.read_sql_query(query, connection)

long_term_customers.head(20)

,customerID,tenure,Contract,MonthlyCharges,TotalCharges,Churn
0,5248-YGIJN,72,Two year,90.25,6369.45,No
1,6234-RAAPL,72,Two year,99.90,7251.70,No
2,5954-BDFSG,72,Two year,107.50,7853.70,No
3,0526-SXDJP,72,Two year,42.10,2962.00,No
4,9848-JQJTX,72,Two year,100.90,7459.05,No
5,6728-DKUCO,72,One year,104.15,7303.05,No
6,2848-YXSMW,72,Two year,19.40,1363.25,No
7,6734-PSBAW,72,Two year,23.55,1723.95,No
8,3146-MSEGF,72,Two year,88.05,6425.65,No
9,5997-OPVFA,72,Two year,89.05,6254.45,No


In [17]:
#High monthly-charge customers
query = """
SELECT
    customerID,
    tenure,
    Contract,
    MonthlyCharges,
    TotalCharges,
    Churn
FROM customers
WHERE MonthlyCharges >= 90
ORDER BY MonthlyCharges DESC;
"""

pd.read_sql_query(query, connection).head(20)

,customerID,tenure,Contract,MonthlyCharges,TotalCharges,Churn
0,7569-NMZYQ,72,Two year,118.75,8672.45,No
1,8984-HPEMB,71,Two year,118.65,8477.60,No
2,5989-AXPUC,68,Two year,118.60,7990.05,No
3,5734-EJKXG,61,One year,118.60,7365.70,No
4,8199-ZLLSA,67,One year,118.35,7804.15,Yes
5,9924-JPRMC,72,Two year,118.20,8547.15,No
6,2889-FPWRM,72,One year,117.80,8684.80,Yes
7,3810-DVDQQ,72,Two year,117.60,8308.90,No
8,9739-JLPQJ,72,Two year,117.50,8670.10,No
9,2302-ANTDP,48,Month-to-month,117.45,5438.90,Yes


In [18]:
query = """
SELECT
    CustomerSegment,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate,
    ROUND(AVG(MonthlyCharges), 2) AS avg_monthly_charges
FROM customers
GROUP BY CustomerSegment
ORDER BY churn_rate DESC;
"""

pd.read_sql_query(query, connection)

,CustomerSegment,total_customers,churned_customers,churn_rate,avg_monthly_charges
0,New High-Value,122,93,76.23,95.64
1,New Customer,1359,691,50.85,51.07
2,Regular Customer,3259,863,26.48,63.02
3,Loyal High-Value,928,158,17.03,103.23
4,Long-Term Customer,1375,64,4.65,53.73


In [19]:
query = """
SELECT
    NumberOfServices,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate,
    ROUND(AVG(MonthlyCharges), 2) AS avg_monthly_charges
FROM customers
GROUP BY NumberOfServices
ORDER BY NumberOfServices;
"""

pd.read_sql_query(query, connection)

,NumberOfServices,total_customers,churned_customers,churn_rate,avg_monthly_charges
0,0,80,35,43.75,24.90
1,1,1701,359,21.11,30.08
2,2,1188,390,32.83,51.13
3,3,965,352,36.48,69.36
4,4,922,289,31.34,78.73
5,5,908,232,25.55,86.31
6,6,676,152,22.49,93.64
7,7,395,49,12.41,99.43
8,8,208,11,5.29,104.63


In [26]:
customer_id = df["customerID"].iloc[0]

query = """
SELECT
    customerID,
    gender,
    SeniorCitizen,
    Partner,
    Dependents,
    tenure,
    PhoneService,
    InternetService,
    OnlineSecurity,
    OnlineBackup,
    DeviceProtection,
    TechSupport,
    StreamingTV,
    StreamingMovies,
    Contract,
    PaperlessBilling,
    PaymentMethod,
    MonthlyCharges,
    TotalCharges,
    Churn,
    CustomerSegment,
    NumberOfServices
FROM customers
WHERE customerID = ?;
"""

customer_profile = pd.read_sql_query(
    query,
    connection,
    params=(customer_id,)
)

customer_profile.T

,0
customerID,7590-VHVEG
gender,Female
SeniorCitizen,0
Partner,Yes
Dependents,No
tenure,1
PhoneService,No
InternetService,DSL
OnlineSecurity,No
OnlineBackup,Yes


In [22]:
import os
os.makedirs("processed", exist_ok=True)

high_risk_customers.to_csv(
    "processed/high_risk_customers.csv",
    index=False
)

print("SQL analysis results saved.")

SQL analysis results saved.
